# FreshCart ELT on Snowflake (local VS Code version)
Mock API data → raw landing files → Snowflake **bronze** → SQL → **silver** (+ **quarantine**) → **gold**, then quality checks.

Based on the course's reference notebook (`freshcart_elt_simple_(1).ipynb`) and flowchart. What is different here:

| Part | Where it lives |
|---|---|
| SQL (`SQL_00_SETUP` … `SQL_40_GOLD`) | `elt/sql/*.sql` (loaded into the constants of the same name) |
| Python: `write_landing`, `load_file`, `quality_checks`, `run_pipeline` | `elt/freshcart.py` |
| Sample ("mock") data | `elt/mock_data.py`, copied from the reference notebook; there are no real APIs |
| Snowflake login | `.env` in the project root (git-ignored), never in the notebook |

Database: `FRESHCART_DB` (the reference uses `FRESHCART`). Run the cells from top to bottom.

## 1. Check the environment

In [1]:
# Dependencies come from requirements-elt.txt (installed once into the project's .venv).
import sys
import snowflake.connector
print("python", sys.version.split()[0], "| snowflake-connector", snowflake.connector.__version__)

python 3.12.6 | snowflake-connector 3.12.4


## 2. Log in to Snowflake
`fc.connect()` reads `SNOWFLAKE_ACCOUNT`, `SNOWFLAKE_USER`, `SNOWFLAKE_PASSWORD` (and role, warehouse, database) from `.env`.
Only the role, warehouse and database are printed.

In [2]:
import freshcart as fc
fc.connect()

Connected: role=ACCOUNTADMIN warehouse=FRESHCART_WH database=FRESHCART_DB


## 3. The sample data (MOCK)
Eight raw order rows (6 from the website, 2 from the mobile app), plus customer tier changes, exchange rates and pincodes.
**This is mock data** taken from the reference notebook. The mess is deliberate: the SQL has to clean it.

In [3]:
import pandas as pd
print("web orders   :", len(fc.MOCK_WEB_ORDERS))
print("mobile orders:", len(fc.MOCK_MOBILE_ORDERS))
print("crm events   :", len(fc.MOCK_CRM_TIER_EVENTS))
print("fx replies   :", len(fc.MOCK_FX_RESPONSES))
pd.DataFrame(fc.MOCK_WEB_ORDERS)

web orders   : 6
mobile orders: 2
crm events   : 4
fx replies   : 1


,order_id,cust_email,cust_name,order_date,amount,currency,pincode,updated_at
0,ORD-1001,Priya.R@Gmail.com,priya r,2026-09-01,"1,250.00",INR,600041,2026-09-01 10:02:00
1,ORD-1003,meera@outlook.com,MEERA S,2026-09-02,None,INR,400050,2026-09-02 09:10:00
2,ORD-1002,Arjun.K@Yahoo.in,Arjun K,2026-09-01,42.50,USD,560034,2026-09-01 11:15:00
3,ORD-1004,rahul.m@gmail.com,Rahul M,"Sep 3, 2026",-500,INR,110001,2026-09-03 12:00:00
4,ORD-1005,priya.r@gmail.com,Priya R.,2026-09-03,₹ 899,inr,600041,2026-09-03 16:45:00
5,ORD-1006,vikram@@mail,Vikram,2026-09-04,30,EUR,99999,2026-09-04 08:30:00


## 4. EXTRACT: save the data as raw files
`fc.write_landing(source, records, run_id)` writes one compressed `.json.gz` file per source (in `elt/landing/`). Nothing is cleaned.
Each record is wrapped in a small envelope:
```
{"source": "web", "run_id": "...", "extracted_at": "...", "data": { <the original record> }}
```

## 5. LOAD: copy a raw file into a bronze table
`fc.load_file(local_path, table, stage_dir)`: `PUT` uploads the file into the Snowflake stage `@bronze.landing`, `COPY INTO` reads it into a bronze table as raw JSON.

## 6. TRANSFORM: the SQL
Each script lives in `elt/sql/` and is loaded into a constant. Nothing runs yet; `run_pipeline()` sends them to Snowflake in this order.

In [4]:
for name in ["SQL_00_SETUP", "SQL_10_ORDERS_CLEAN", "SQL_20_ORDERS_DEDUP", "SQL_30_CUSTOMERS_SCD2",
             "SQL_31_FX_DAILY", "SQL_32_PINCODE_GEO", "SQL_40_GOLD"]:
    sql = getattr(fc, name)
    print(f"{name:24s} {len(sql.splitlines()):3d} lines")

SQL_00_SETUP              97 lines
SQL_10_ORDERS_CLEAN       73 lines
SQL_20_ORDERS_DEDUP       11 lines
SQL_30_CUSTOMERS_SCD2     22 lines
SQL_31_FX_DAILY           37 lines
SQL_32_PINCODE_GEO        25 lines
SQL_40_GOLD               47 lines


## 7. The pipeline
`fc.run_pipeline()` does the four stages in order: extract, load, transform, check.

## 8. Run it
First the setup script: it creates the schemas, stage, file format and empty tables (safe to run again).

In [5]:
fc.run_sql(fc.SQL_00_SETUP)
print("Setup done")

Setup done


**Before the first run**, confirm that bronze is empty. The B1 numbers (8 raw order rows) are only valid after exactly one run.

In [6]:
before = fc.query("SELECT COUNT(*) FROM bronze.orders_raw")[0][0]
print("bronze.orders_raw rows before the run:", before)
assert before == 0, "bronze is not empty: the B1 baseline needs a fresh start"

bronze.orders_raw rows before the run:

 0


Now the pipeline. Expected last line: **GOLD: 4 orders, revenue ₹7785.00, 126 loyalty points**. "Share of rows rejected: 25%" is correct for this sample (2 bad rows out of 8).

In [7]:
run_info = fc.run_pipeline()

run_id: 20261112_040919
1. EXTRACT: sample data -> raw files
   web: 6 records -> elt\landing\web\web__20261112_040919.json.gz
   mobile: 2 records -> elt\landing\mobile\mobile__20261112_040919.json.gz
   crm: 4 records -> elt\landing\crm\crm__20261112_040919.json.gz
   fx: 1 records -> elt\landing\fx\fx__20261112_040919.json.gz
2. LOAD: raw files -> bronze tables


   6 rows -> bronze.orders_raw


   2 rows -> bronze.orders_raw


   4 rows -> bronze.customers_raw


   1 rows -> bronze.fx_raw
3. TRANSFORM: bronze -> silver -> gold


   new pincodes to look up: ['600041', '560034', '400050']
   pincode: 3 records -> elt\landing\pincode\pincode__20261112_040919.json.gz


... (3 outputs left out)

   orders with no customer match          0   OK
   orders with an amount but no FX rate   0   OK


   orders with no city                    0   OK


   share of rows rejected                 25%


GOLD: 4 orders, revenue ₹7785.00, 126 loyalty points


## 9. Look inside each layer
### Bronze: raw JSON exactly as received

In [8]:
fc.show("SELECT ingest_seq, source, raw FROM bronze.orders_raw ORDER BY ingest_seq")

,INGEST_SEQ,SOURCE,RAW
0,1,web,"{\n ""amount"": ""1,250.00"",\n ""currency"": ""INR..."
1,2,web,"{\n ""amount"": null,\n ""currency"": ""INR"",\n ..."
2,3,web,"{\n ""amount"": ""42.50"",\n ""currency"": ""USD"",\..."
3,4,web,"{\n ""amount"": ""-500"",\n ""currency"": ""INR"",\n..."
4,5,web,"{\n ""amount"": ""₹ 899"",\n ""currency"": ""inr"",\..."
5,6,web,"{\n ""amount"": 30,\n ""currency"": ""EUR"",\n ""c..."
6,7,mobile,"{\n ""customer"": {\n ""email"": ""arjun.k@yaho..."
7,8,mobile,"{\n ""customer"": {\n ""email"": ""Meera@Outloo..."


### Silver: cleaned rows (6)

In [9]:
fc.show("SELECT ingest_seq, source, order_id, email, name, order_date, amount, currency, pincode, amount_missing FROM silver.orders_clean ORDER BY ingest_seq")

,INGEST_SEQ,SOURCE,ORDER_ID,EMAIL,NAME,ORDER_DATE,AMOUNT,CURRENCY,PINCODE,AMOUNT_MISSING
0,1,web,ORD-1001,priya.r@gmail.com,Priya R,2026-09-01,1250.00,INR,600041,False
1,2,web,ORD-1003,meera@outlook.com,Meera S,2026-09-02,None,INR,400050,True
2,3,web,ORD-1002,arjun.k@yahoo.in,Arjun K,2026-09-01,42.50,USD,560034,False
3,5,web,ORD-1005,priya.r@gmail.com,Priya R,2026-09-03,899.00,INR,600041,False
4,7,mobile,ORD-1002,arjun.k@yahoo.in,Arjun K,2026-09-01,42.50,USD,560034,False
5,8,mobile,ORD-1003,meera@outlook.com,Meera S,2026-09-02,2100.00,INR,400050,False


### Quarantine: rejected rows and why

In [10]:
fc.show("SELECT run_id, source, order_id, reasons FROM quarantine.orders ORDER BY order_id")

,RUN_ID,SOURCE,ORDER_ID,REASONS
0,20261112_040919,web,ORD-1004,"[\n ""non_positive_amount""\n]"
1,20261112_040919,web,ORD-1006,"[\n ""bad_email"",\n ""bad_pincode""\n]"


### Silver: one row per order (4)

In [11]:
fc.show("SELECT order_id, ingest_seq, updated_at, amount FROM silver.orders_dedup ORDER BY order_id")

,ORDER_ID,INGEST_SEQ,UPDATED_AT,AMOUNT
0,ORD-1001,1,2026-09-01 10:02:00,1250.00
1,ORD-1002,7,2026-09-01 11:15:00,42.50
2,ORD-1003,8,2026-09-02 18:40:00,2100.00
3,ORD-1005,5,2026-09-03 16:45:00,899.00


### Dim: customer tier history (SCD2)

In [12]:
fc.show("SELECT * FROM dim.customers_scd2 ORDER BY cust_id, valid_from")

,CUST_ID,EMAIL,TIER,VALID_FROM,VALID_TO,IS_CURRENT
0,CUST-01,priya.r@gmail.com,Silver,2023-04-10,2026-09-02,False
1,CUST-01,priya.r@gmail.com,Gold,2026-09-03,9999-12-31,True
2,CUST-02,arjun.k@yahoo.in,Silver,2024-01-15,9999-12-31,True
3,CUST-03,meera@outlook.com,Platinum,2022-11-01,9999-12-31,True


### Ref: daily FX rates, weekends filled

In [13]:
fc.show("SELECT * FROM ref.fx_daily WHERE currency = 'USD' ORDER BY rate_date LIMIT 10")

,CURRENCY,RATE_DATE,RATE_TO_INR,IS_CARRIED_FORWARD
0,USD,2026-09-01,83.2000,False
1,USD,2026-09-02,83.3500,False
2,USD,2026-09-03,83.1000,False
3,USD,2026-09-04,83.1000,True
4,USD,2026-09-05,83.1000,True
5,USD,2026-09-06,83.1000,True
6,USD,2026-09-07,83.1000,True
7,USD,2026-09-08,83.1000,True
8,USD,2026-09-09,83.1000,True
9,USD,2026-09-10,83.1000,True


### Ref: pincode geography

In [14]:
fc.show("SELECT * FROM ref.pincode_geo ORDER BY pincode")

,PINCODE,CITY,STATE,REGION,UPDATED_AT
0,400050,Mumbai,Maharashtra,West,2026-11-11 20:09:27.889000-08:00
1,560034,Bengaluru,Karnataka,South,2026-11-11 20:09:27.889000-08:00
2,600041,Chennai,Tamil Nadu,South,2026-11-11 20:09:27.889000-08:00


### Gold: the final table

In [15]:
fc.show("SELECT order_id, order_date, tier, amount, currency, fx_rate, amount_inr, loyalty_points, is_high_value, city, region FROM gold.fct_orders ORDER BY order_id")

,ORDER_ID,ORDER_DATE,TIER,AMOUNT,CURRENCY,FX_RATE,AMOUNT_INR,LOYALTY_POINTS,IS_HIGH_VALUE,CITY,REGION
0,ORD-1001,2026-09-01,Silver,1250.00,INR,1.0000,1250.00,12,False,Chennai,South
1,ORD-1002,2026-09-01,Silver,42.50,USD,83.2000,3536.00,35,True,Bengaluru,South
2,ORD-1003,2026-09-02,Platinum,2100.00,INR,1.0000,2100.00,63,True,Mumbai,West
3,ORD-1005,2026-09-03,Gold,899.00,INR,1.0000,899.00,16,False,Chennai,South


### Gold totals

In [16]:
fc.show("SELECT COUNT(*) AS orders, SUM(amount_inr) AS revenue_inr, SUM(loyalty_points) AS points FROM gold.fct_orders")

,ORDERS,REVENUE_INR,POINTS
0,4,7785.00,126


## 10. B1 check: expected vs actual (after exactly one run)
Expected values come from the assignment table (B1) and the flowchart. `PASS` only if the actual number matches.

In [17]:
q = lambda sql: fc.query(sql)[0][0]
rejected_share = float(q("SELECT COUNT_IF(ARRAY_SIZE(reasons) > 0) / NULLIF(COUNT(*), 0) FROM silver.orders_checked"))
quarantined = sorted(r[0] for r in fc.query("SELECT order_id FROM quarantine.orders"))
gold_rev = q("SELECT SUM(amount_inr) FROM gold.fct_orders")
rows = [
    ("Raw order rows in bronze.orders_raw", 8, q("SELECT COUNT(*) FROM bronze.orders_raw")),
    ("  of which web", 6, q("SELECT COUNT(*) FROM bronze.orders_raw WHERE source = 'web'")),
    ("  of which mobile", 2, q("SELECT COUNT(*) FROM bronze.orders_raw WHERE source = 'mobile'")),
    ("Rows in bronze.customers_raw (tier changes)", 4, q("SELECT COUNT(*) FROM bronze.customers_raw")),
    ("Rows in bronze.fx_raw (FX replies)", 1, q("SELECT COUNT(*) FROM bronze.fx_raw")),
    ("Rows in bronze.pincode_raw (3 on the first run)", 3, q("SELECT COUNT(*) FROM bronze.pincode_raw")),
    ("Rows that pass cleaning (silver.orders_clean)", 6, q("SELECT COUNT(*) FROM silver.orders_clean")),
    ("Rows in quarantine.orders", 2, q("SELECT COUNT(*) FROM quarantine.orders")),
    ("Quarantined order ids", ["ORD-1004", "ORD-1006"], quarantined),
    ("Orders after deduplication (silver.orders_dedup)", 4, q("SELECT COUNT(*) FROM silver.orders_dedup")),
    ("Rows in dim.customers_scd2 (tiers)", 4, q("SELECT COUNT(*) FROM dim.customers_scd2")),
    ("Rows in ref.pincode_geo", 3, q("SELECT COUNT(*) FROM ref.pincode_geo")),
    ("Gold orders (gold.fct_orders)", 4, q("SELECT COUNT(*) FROM gold.fct_orders")),
    ("Gold revenue (INR)", "7785.00", f"{gold_rev:.2f}"),
    ("Gold loyalty points", 126, q("SELECT SUM(loyalty_points) FROM gold.fct_orders")),
    ("Share of rows rejected", "25%", f"{rejected_share:.0%}"),
]
result = pd.DataFrame([(n, e, a, "PASS" if str(e) == str(a) else "FAIL") for n, e, a in rows],
                      columns=["Check", "Expected", "Actual", "Result"])
print(f"{(result.Result == 'PASS').sum()} of {len(result)} checks PASS")
result

16 of 16 checks PASS


,Check,Expected,Actual,Result
0,Raw order rows in bronze.orders_raw,8,8,PASS
1,of which web,6,6,PASS
2,of which mobile,2,2,PASS
3,Rows in bronze.customers_raw (tier changes),4,4,PASS
4,Rows in bronze.fx_raw (FX replies),1,1,PASS
5,Rows in bronze.pincode_raw (3 on the first run),3,3,PASS
6,Rows that pass cleaning (silver.orders_clean),6,6,PASS
7,Rows in quarantine.orders,2,2,PASS
8,Quarantined order ids,"[ORD-1004, ORD-1006]","[ORD-1004, ORD-1006]",PASS
9,Orders after deduplication (silver.orders_dedup),4,4,PASS


## 11. Run it again: nothing should double *(not run yet)*
Bronze grows by 8 order rows on every run, because it keeps every copy it is given. Quarantine and gold stay the same.
This section is run later (B3/B4), so the B1 evidence above is taken after exactly one run.

In [ ]:
fc.run_pipeline()
fc.show("""
SELECT (SELECT COUNT(*) FROM bronze.orders_raw)  AS bronze_rows,
       (SELECT COUNT(*) FROM quarantine.orders)  AS quarantine_rows,
       (SELECT COUNT(*) FROM gold.fct_orders)    AS gold_orders,
       (SELECT SUM(amount_inr) FROM gold.fct_orders) AS revenue_inr,
       (SELECT SUM(loyalty_points) FROM gold.fct_orders) AS points
""")

## 12. Clean up (optional)
Removes everything this notebook created in Snowflake. Uncomment to run.

In [ ]:
# fc.query("DROP DATABASE IF EXISTS FRESHCART_DB")
# fc.query("DROP WAREHOUSE IF EXISTS FRESHCART_WH")
fc.conn.close()